# RetailIQ: Data Exploration & Visualization

**Project**: E-commerce Customer Purchase Prediction  
**Dataset**: 2.7M e-commerce events  
**Goal**: Understand the data and visualize patterns

## 1. Setup
Import libraries and initialize Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, hour, from_unixtime
import matplotlib.pyplot as plt
import pandas as pd

# Start Spark
spark = SparkSession.builder \
    .appName("RetailIQ_DataExploration") \
    .master("local[*]") \
    .getOrCreate()

print("✓ Spark initialized")

## 2. Load Dataset

In [ ]:
# Load data
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("../data/events.csv")

print(f"Total Records: {df.count():,}")
print(f"\nColumns: {df.columns}")

## 3. Data Overview

In [ ]:
# Show sample data
df.show(10)

In [ ]:
# Data schema
df.printSchema()

## 4. Event Distribution

In [ ]:
# Count events by type
event_counts = df.groupBy("event") \
    .count() \
    .orderBy("count", ascending=False) \
    .toPandas()

print("Event Counts:")
print(event_counts)

In [ ]:
# Visualize event distribution
plt.figure(figsize=(10, 6))
plt.bar(event_counts['event'], event_counts['count'], color=['#3498db', '#e74c3c', '#2ecc71'])
plt.title('Event Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Event Type')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)

# Add count labels
for i, v in enumerate(event_counts['count']):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 5. Visitor Engagement

In [ ]:
# Events per visitor
visitor_events = df.groupBy("visitorid").agg(count("*").alias("event_count"))

# Categorize engagement
engagement = visitor_events.withColumn(
    "category",
    when(col("event_count") == 1, "1 event")
    .when(col("event_count") <= 5, "2-5 events")
    .when(col("event_count") <= 10, "6-10 events")
    .when(col("event_count") <= 20, "11-20 events")
    .otherwise("20+ events")
)

engagement_counts = engagement.groupBy("category").count().toPandas()
print(engagement_counts)

In [ ]:
# Visualize engagement
categories = ["1 event", "2-5 events", "6-10 events", "11-20 events", "20+ events"]
engagement_counts = engagement_counts.set_index('category').reindex(categories).reset_index()

plt.figure(figsize=(10, 6))
plt.bar(engagement_counts['category'], engagement_counts['count'], color='#3498db')
plt.title('Visitor Engagement Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Events per Visitor')
plt.ylabel('Number of Visitors')
plt.xticks(rotation=15)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Hourly Activity Pattern

In [ ]:
# Extract hour from timestamp
hourly_df = df.withColumn("hour", hour(from_unixtime(col("timestamp") / 1000)))

hourly_counts = hourly_df.groupBy("hour") \
    .count() \
    .orderBy("hour") \
    .toPandas()

print("Hourly Activity:")
print(hourly_counts.head(10))

In [ ]:
# Visualize hourly pattern
plt.figure(figsize=(12, 6))
plt.plot(hourly_counts['hour'], hourly_counts['count'], marker='o', linewidth=2, color='#e74c3c')
plt.title('E-commerce Activity by Hour', fontsize=16, fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Events')
plt.xticks(range(0, 24))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Top Products

In [ ]:
# Top viewed products
top_viewed = df.filter(col("event") == "view") \
    .groupBy("itemid") \
    .count() \
    .orderBy("count", ascending=False) \
    .limit(10) \
    .toPandas()

print("Top 10 Viewed Products:")
print(top_viewed)

In [ ]:
# Visualize top products
plt.figure(figsize=(10, 6))
plt.barh(top_viewed['itemid'].astype(str), top_viewed['count'], color='#2ecc71')
plt.gca().invert_yaxis()
plt.title('Top 10 Most Viewed Products', fontsize=16, fontweight='bold')
plt.xlabel('Number of Views')
plt.ylabel('Product ID')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(f"Total Events: {df.count():,}")
print(f"Unique Visitors: {df.select('visitorid').distinct().count():,}")
print(f"Unique Products: {df.select('itemid').distinct().count():,}")
print(f"Event Types: {df.select('event').distinct().count()}")
print("=" * 50)

## 9. Stop Spark

In [ ]:
spark.stop()
print("✓ Spark stopped")

---
## Key Insights

1. **Most events are views** - customers browse a lot
2. **Few transactions** - low conversion rate
3. **Peak activity hours** - can identify best times for marketing
4. **Popular products** - can focus inventory on these items
5. **Visitor engagement varies** - some visit once, others are very engaged

**Next**: Build ML models to predict which visitors will make purchases!